In [2]:
import pandas as pd
df = pd.read_csv("ventas_2025.csv")
df.head()

,fecha,ciudad,categoria,producto,canal,cantidad,precio_unitario,total_venta,cliente_id,vendedor
0,2025-06-14,Cali,Deportes,Balón de fútbol,Online,1,776368.02,776368.02,CLI-2375,Carlos
1,2025-06-16,Cali,Moda,Jeans,Tienda,2,2650654.73,5301309.46,CLI-2237,Carlos
2,2025-07-12,Tunja,Tecnología,Audífonos,Tienda,3,3176430.30,9529290.90,CLI-9296,Felipe
3,2025-11-27,Bogotá,Entretenimiento,Control de videojuego,Tienda,3,2684875.51,8054626.53,CLI-3200,Camila
4,2025-03-14,Cali,Deportes,Tenis de correr,App,3,100278.87,300836.61,CLI-7437,Laura


In [19]:
doc = df.to_dict("records")
print("Cantidad de documentos: ", len(doc))
print("Tipo de cada documento: ", type(doc[0]))

Cantidad de documentos:  3008
Tipo de cada documento:  <class 'dict'>


In [20]:
import json
print(json.dumps(doc[0], indent=2, ensure_ascii=False))

{
  "fecha": "2025-06-14",
  "ciudad": "Cali",
  "categoria": "Deportes",
  "producto": "Balón de fútbol",
  "canal": "Online",
  "cantidad": 1,
  "precio_unitario": 776368.02,
  "total_venta": 776368.02,
  "cliente_id": "CLI-2375",
  "vendedor": "Carlos"
}


In [21]:
doc_con_extra = dict(doc[0])
doc_con_extra["descuento"] = 0.15
print(json.dumps([doc_con_extra], indent=2, ensure_ascii=False))

[
  {
    "fecha": "2025-06-14",
    "ciudad": "Cali",
    "categoria": "Deportes",
    "producto": "Balón de fútbol",
    "canal": "Online",
    "cantidad": 1,
    "precio_unitario": 776368.02,
    "total_venta": 776368.02,
    "cliente_id": "CLI-2375",
    "vendedor": "Carlos",
    "descuento": 0.15
  }
]


Operaciones Crud

In [22]:
#Crear
venta_nueva = {
    "fecha": "2026-01-10",
    "ciudad": "Medellín",
    "categoria": "Tecnología",
    "producto": "Laptop",
    "canal": "Online",
    "cantidad": 2,
    "precio_unitario": 2500000,
    "total_venta": 60000000000,
    "cliente_id": "CLI-NUEVO",
    "vendedor": "Laura",
}

doc.append(venta_nueva)
print("Documentos despupes de crear un nuevo registro: ", len(doc))
print("Ultimo registro en el documento: " )
print(json.dumps(doc[-1], indent=2, ensure_ascii=False))

Documentos despupes de crear un nuevo registro:  3009
Ultimo registro en el documento: 
{
  "fecha": "2026-01-10",
  "ciudad": "Medellín",
  "categoria": "Tecnología",
  "producto": "Laptop",
  "canal": "Online",
  "cantidad": 2,
  "precio_unitario": 2500000,
  "total_venta": 60000000000,
  "cliente_id": "CLI-NUEVO",
  "vendedor": "Laura"
}


In [ ]:
#Read
filtrar = [doc for doc in documentos if doc["ciudad"] == "Medellín"]
print("ventas flitros ", len(filtrar)) 
print(json.dumps(filtrar[0], indent=2, ensure_ascii=False))

ventas flitros  486


TypeError: Object of type ObjectId is not JSON serializable

In [ ]:
filtrar = [doc for doc in documentos if doc["cliente_id"] == "CLI-NUEVO"]
print("ventas flitros ", len(filtrar)) 
print(json.dumps(filtrar[0], indent=2, ensure_ascii=False))

ventas flitros  1
{
  "fecha": "2026-01-10",
  "ciudad": "Medellín",
  "categoria": "Tecnología",
  "producto": "Laptop",
  "canal": "Online",
  "cantidad": 2,
  "precio_unitario": 2500000,
  "total_venta": 60000000000,
  "cliente_id": "CLI-NUEVO",
  "vendedor": "Laura"
}


In [ ]:
#Update
for doc in documentos:
    if doc["cliente_id"] == "CLI-NUEVO":
        doc["canal"] = "App"
        break
print(json.dumps(filtrar[0], indent=2, ensure_ascii=False))

{
  "fecha": "2026-01-10",
  "ciudad": "Medellín",
  "categoria": "Tecnología",
  "producto": "Laptop",
  "canal": "App",
  "cantidad": 2,
  "precio_unitario": 2500000,
  "total_venta": 60000000000,
  "cliente_id": "CLI-NUEVO",
  "vendedor": "Laura"
}


In [ ]:
#Eliminar
documentos = [doc for doc in documentos if doc["cliente_id"] != "CLI-NUEVO"]
print("Documento después de borrar: ", len(documentos)) 

Documento después de borrar:  3008


In [ ]:
import pymongo
print("pymongo", pymongo.version)
df = pd.read_csv('ventas_2025.csv')

pymongo 4.17.0


In [11]:
try:
    from pymongo import MongoClient
    client = MongoClient('mongodb://localhost:27017/', serverSelectionTimeoutMS=2000)
    client.admin.command('ping')  # sólo si el servidor responde
    MODO = 'Servidor MongoDB real'
except Exception as e:
    import mongomock
    client = mongomock.MongoClient()
    MODO = 'MongoDB simulado (mongomock) porque no hay servidor corriendo'

print('Modo conexión:', MODO)

# Creamos la base de datos y la colección (como una tabla, pero de documentos)
db = client.bigData
coleccion = db.ventas
print('Base de datos: bigdata | Colección: ventas')

Modo conexión: Servidor MongoDB real
Base de datos: bigdata | Colección: ventas


In [12]:
coleccion.delete_many({})

documentos = df.to_dict('records')
resultado = coleccion.insert_many(documentos)
print('Documentos insertados:', len(resultado.inserted_ids))
print('Total de documentos en la colección:', coleccion.count_documents({}))

Documentos insertados: 3008
Total de documentos en la colección: 3008


In [13]:
ventas_cali = coleccion.find({"ciudad": "Cali"})
lista_cali = list(ventas_cali)
print("Ventas en Cali: ", len(lista_cali))
print("Primera venta Cali: ")
print(lista_cali[0])

Ventas en Cali:  463
Primera venta Cali: 
{'_id': ObjectId('6a97008f331475ccbe5c3382'), 'fecha': '2025-06-14', 'ciudad': 'Cali', 'categoria': 'Deportes', 'producto': 'Balón de fútbol', 'canal': 'Online', 'cantidad': 1, 'precio_unitario': 776368.02, 'total_venta': 776368.02, 'cliente_id': 'CLI-2375', 'vendedor': 'Carlos'}


In [14]:
filtro = {"ciudad": "Bogotá", "categoria": "Tecnología", "canal": "Online"}
print("Tecnología Online en Bogotá: ", coleccion.count_documents(filtro))

for doc in list(coleccion.find(filtro).limit(3)):
    print(doc["producto"], "-", doc["total_venta"])

Tecnología Online en Bogotá:  25
Audífonos - 3492676.78
Smartwatch - 591235.16
Smartphone - 889144.5


In [15]:
venta_nueva= {
    "fecha" : "2026-03-01", "ciudad": "Bucaramanga",
    "categoria": "Tecnología", "producto": "Tablet", "canal": "Online",
    "cantidad": 1, "precio_unitario": "1500000.0", "total_venta": 1500000.0,
    "cliente_id": "CLI_MIA", "vendedor": "Sofia"
}
res = coleccion.insert_one(venta_nueva)
print("Documento insertado con id: ", res.inserted_id)
print("Total de documentos ahora: ", coleccion.count_documents({}))

Documento insertado con id:  6a9702c2331475ccbe5c3f42
Total de documentos ahora:  3009


In [26]:
res = coleccion.update_one(
    {"cliente_id": "CLI_MIA"},
    {"$set": {"canal": "Tienda"}}
)
print("Documentos modificados: ", res.modified_count)

print("Documento actualizado: ")
print(coleccion.find_one({"cliente_id": "CLI_MIA"}))

Documentos modificados:  0
Documento actualizado: 
{'_id': ObjectId('6a9702c2331475ccbe5c3f42'), 'fecha': '2026-03-01', 'ciudad': 'Bucaramanga', 'categoria': 'Tecnología', 'producto': 'Tablet', 'canal': 'Tienda', 'cantidad': 1, 'precio_unitario': '1500000.0', 'total_venta': 1500000.0, 'cliente_id': 'CLI_MIA', 'vendedor': 'Sofia'}


In [27]:
res = coleccion.delete_many({"cliente_id": "CLI_MIA"})
print("Documentos eliminados: ", res.deleted_count)
print("Total de documentos ahora: ", coleccion.count_documents({}))

Documentos eliminados:  1
Total de documentos ahora:  3008


Ejercicio 1: ¿Cuántas ventas hay en Bogotá? (find + count_documents)

In [28]:
total_bogota = coleccion.count_documents({"ciudad": "Bogotá"})
print("Ventas en Bogotá: ", total_bogota)

Ventas en Bogotá:  485


Ejercicio 2: ¿Cuántas ventas de la categoría Hogar son por canal App? (dos condiciones)

In [29]:
total_hogar_app = coleccion.count_documents({"categoria": "Hogar", "canal": "App"})
print("Ventas de Hogar por canal App: ", total_hogar_app)

Ventas de Hogar por canal App:  186


Ejercicio 3: ¿Cuántas ventas superan $2.000.000? (operador $gt)

In [30]:
total_mayor_2m = coleccion.count_documents({"total_venta": {"$gt": 2000000}})
print("Ventas que superan $2.000.000: ", total_mayor_2m)

Ventas que superan $2.000.000:  1991


Ejercicio 4: Inserta una venta nueva en la colección (inventa los datos) y verifica que quedó

In [31]:
venta_ejercicio4 = {
    "fecha": "2026-02-15",
    "ciudad": "Cartagena",
    "categoria": "Hogar",
    "producto": "Licuadora",
    "canal": "Online",
    "cantidad": 1,
    "precio_unitario": 250000.0,
    "total_venta": 250000.0,
    "cliente_id": "CLI-EJ4",
    "vendedor": "Camila"
}
res = coleccion.insert_one(venta_ejercicio4)
print("Documento insertado con id: ", res.inserted_id)
print("Verificación: ", coleccion.find_one({"cliente_id": "CLI-EJ4"}))


Documento insertado con id:  6a970797331475ccbe5c3f43
Verificación:  {'_id': ObjectId('6a970797331475ccbe5c3f43'), 'fecha': '2026-02-15', 'ciudad': 'Cartagena', 'categoria': 'Hogar', 'producto': 'Licuadora', 'canal': 'Online', 'cantidad': 1, 'precio_unitario': 250000.0, 'total_venta': 250000.0, 'cliente_id': 'CLI-EJ4', 'vendedor': 'Camila'}


Ejercicio 5: Actualiza la venta que insertaste: cambia su canal a Tienda

In [32]:
res = coleccion.update_one(
    {"cliente_id": "CLI-EJ4"},
    {"$set": {"canal": "Tienda"}}
)
print("Documentos modificados: ", res.modified_count)
print("Documento actualizado: ")
print(coleccion.find_one({"cliente_id": "CLI-EJ4"}))


Documentos modificados:  1
Documento actualizado: 
{'_id': ObjectId('6a970797331475ccbe5c3f43'), 'fecha': '2026-02-15', 'ciudad': 'Cartagena', 'categoria': 'Hogar', 'producto': 'Licuadora', 'canal': 'Tienda', 'cantidad': 1, 'precio_unitario': 250000.0, 'total_venta': 250000.0, 'cliente_id': 'CLI-EJ4', 'vendedor': 'Camila'}


Ejercicio 6: Elimina la venta que insertaste y verifica que ya no está

In [33]:
res = coleccion.delete_many({"cliente_id": "CLI-EJ4"})
print("Documentos eliminados: ", res.deleted_count)
print("¿Sigue existiendo?: ", coleccion.find_one({"cliente_id": "CLI-EJ4"}))


Documentos eliminados:  1
¿Sigue existiendo?:  None


Ejercicio 7 (bonus): Cuenta las ventas de Cali o Medellín con total menor a $1.000.000 ($in + $lt)

In [34]:
total_cali_medellin_bajo = coleccion.count_documents({
    "ciudad": {"$in": ["Cali", "Medellín"]},
    "total_venta": {"$lt": 1000000}
})
print("Ventas de Cali o Medellín con total < $1.000.000: ", total_cali_medellin_bajo)


Ventas de Cali o Medellín con total < $1.000.000:  160
